# D — интерактивная симуляция с активностью нейронов, афферентов и силами мышц

Выберите ядро **SpikingNN (.venv)** и выполните единственную кодовую ячейку.
Это отдельная расширенная копия ячейки из `explore_genome.ipynb`.
Слева **A** — стержень, мышцы и силы; справа **B** — целевой и фактический углы,
**C** — мембранные потенциалы N0/N1 или растр их спайков, **D** — сумма
откликов Ia + II + Ib отдельно для сгибателя и разгибателя, **E** — силы
натяжения мышц в ньютонах (сгибатель — красный, разгибатель — синий).
Четыре временных графика справа занимают ту же общую высоту.
Все они используют общее скользящее окно `WINDOW_S` (сейчас **0,5 с**).

Кнопка «Нейроны» переключает вид без перезапуска модели. Ползунок, старт/стоп,
сброс и снимок работают как в исходной тетрадке. Снимок сохраняет все панели
в `results/live_activity_snapshots/`. Шрифты и размеры задаются в начале ячейки.

Потенциалы, афференты и силы мышц записываются на каждом шаге **0,1 мс**;
на их графиках используется отображение минимумов/максимумов временных интервалов,
чтобы сохранить пики при ограничении числа отрисовываемых точек. Растр использует
все времена спайков. Афференты показаны в единицах модели, а не в Гц; сумма
Ia + II + Ib — визуальный показатель, не взвешенный ток обратной связи.

Цвет нейрона соответствует его наиболее сильной связи в матрице `P`: сгибатель — красный, разгибатель — синий. Цвет одинаков в режимах потенциалов и растра; он не означает отсутствия связей с другой мышцей.

**English.** Run the single code cell in the **SpikingNN (.venv)** kernel.
Panel A shows the limb, muscles and forces; B shows desired/actual angles;
C switches between the two membrane potentials and an exact spike raster;
D shows the sum of Ia + II + Ib separately for each muscle; E shows muscle
tension in newtons (flexor red, extensor blue). The four time plots fit within
the same total height and share a rolling `WINDOW_S` window (currently 0.5 s).
Voltages, afferents and forces are recorded at every 0.1 ms integration step.
Curves use a min/max display envelope; all spike times are retained. Afferent
sums are in model units, not Hz or the weighted feedback current. Stop/start
preserves the physical state. PNG snapshots include all panels and are saved
under `results/live_activity_snapshots/`. Font and display settings are at the
top of the cell; the original notebook is unchanged.

Neuron colors follow the strongest output projection in `P`: flexor red, extensor blue. Voltage and raster views share these colors; cross-muscle projections remain possible.



<!-- current-training -->
**Переобучение: времена мышц ≥50 мс.** Геном `C_transfer`. Завершено тестов 14/18, строгий успех 0/18. Перезапустите ядро. [Результаты](results/README.md).

**English.** Muscle times ≥50 ms; winner `C_transfer`. Completed 14/18; strict success 0/18. Restart the kernel. [Results](results/README.en.md).


**Проверка этой серии:** резкий переход цели +45° → −45° вызвал падение через 0,5864 с. После остановки используйте «Сброс».

**English:** the +45° → −45° target step caused a fall after 0.5864 s in this study; use Reset after the simulation stops.


<!-- mirror-genome-variants -->

Три сохранённые версии: [быстрые мышцы](best_genome_fast_muscle.json), [времена ≥50 мс](best_genome_slow_muscle.json), [времена ≥50 мс и симметрия](best_genome_slow_muscle_mirrow.json). Выберите версию через `GENOME_VARIANT`; `GENOME_PATH` позволяет указать другой файл. По умолчанию выбран зеркальный геном. Повторный запуск ячейки обновляет старые загрузчики в памяти ядра. [Происхождение и параметры](GENOME_VERSIONS.md).

Three saved variants: fast muscles, ≥50 ms muscles, and ≥50 ms with exact mirror symmetry. Select a named version using `GENOME_VARIANT`, or override it with `GENOME_PATH`. The mirror genome is selected by default. Rerunning the cell refreshes stale cached codecs. [Sources and values](GENOME_VERSIONS.en.md).

In [9]:
# LIVE_ACTIVITY_SIMULATION: run this cell on its own; earlier cells are not required.
# All UI code, configuration, class definition and launch are contained here.
# The existing AngleController supplies the tested neural/muscle/Afferents physics.
import asyncio
from collections import deque
from datetime import datetime
from html import escape
from io import BytesIO
import math
import os
from pathlib import Path
import sys
import time

import ipywidgets as widgets
from IPython.display import display
import matplotlib as mpl
from matplotlib import font_manager
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure
from matplotlib.patches import FancyArrowPatch
import numpy as np

# ---------------- USER SETTINGS: change these and rerun this cell ----------------
# Choose fast_muscle, slow_muscle, slow_muscle_mirrow, or current.
GENOME_VARIANT = "slow_muscle_mirrow"
GENOME_PATH = None              # Optional explicit path; overrides GENOME_VARIANT.
INITIAL_ANGLE_DEG = 0.0         # Actual initial angle relative to the unstable vertical.
INITIAL_SPEED_RAD_S = 0.0       # Actual initial angular velocity.
INITIAL_TARGET_DEG = 0.0        # Slider starts at this desired angle, within [-45, 45].
TARGET_STEP_DEG = 0.5           # Slider resolution; currents are evaluated by the saved polynomial.
WINDOW_S = .5                 # Shared rolling window for angles, neurons, afferents and muscle forces.
COMMAND_INTERVAL_S = 0.01       # Command blocks; neural/afferent signals are recorded every 0.1 ms.
FPS = 15                       # Maximum display refresh rate, independent of the physics step.
TIME_SCALE = 1.0                # 1.0 = one simulated second per wall-clock second.
MAX_BLOCKS_PER_FRAME = 10       # Limit catch-up work so slider/buttons remain responsive.
AUTO_START = os.environ.get('SPIKINGNN_NOTEBOOK_SMOKE') != '1'  # Set False to start manually.

FONT_FAMILY = 'Times New Roman' # Font used for all labels inside the figure.
FONT_SIZE = 14                  # Base figure font size.
TITLE_SIZE = 15                 # Panel titles.
LABEL_SIZE = 14                 # Axis labels.
TICK_SIZE = 13                  # Tick labels.
LEGEND_SIZE = 13                # Legend text.
ANNOTATION_SIZE = 11            # Angles, time and force values printed on the diagram.
WIDGET_FONT_SIZE = 15           # Text size of the HTML status/readout, in pixels.
FIGSIZE = (14, 9)               # Schematic on the left, four synchronized plots on the right; total height is unchanged.
FRAME_DPI = 95                 # Interactive image resolution; lower this if display is slow.
SNAPSHOT_DPI = 220              # Resolution of the PNG saved by the snapshot button.
FORCE_SCALE_M_PER_N = 0.004     # Base display length per newton, before visual length limits.
GRAVITY_MIN_ARROW_M = 0.06      # Minimum visible gravity-arrow length; does not change physical mg.
MAX_ARROW_M = 0.16              # Cap display length only; printed force values remain unchanged.
SNAPSHOT_DIR = None             # None saves under D/results/live_activity_snapshots.
FLEXOR_COLOR = '#d62728'         # Flexor muscle: red.
EXTENSOR_COLOR = '#1764c0'       # Extensor muscle: blue.
TARGET_COLOR = '#d62728'         # Desired direction and desired-angle curve.
LIMB_COLOR = '#000000'           # Actual limb and actual-angle curve.
# Neural colors follow the strongest P[muscle, neuron] projection in the loaded genome:
# flexor-dominant neurons are red; extensor-dominant neurons are blue.
NEURON_VIEW = 'membrane'         # Initial view: 'membrane' (mV) or 'spikes' (event raster).
MAX_DISPLAY_POINTS = 2400        # Min/max display envelope per curve; full samples remain in memory.


def _live_project_root():
    """Locate the project from the notebook directory or repository root."""
    for parent in (Path.cwd(), *Path.cwd().parents):
        for candidate in (parent, parent / 'scripts/AfferentBalance2026'):
            if (candidate / 'experiments/D_trajectory/genome_cfg.json').is_file():
                return candidate.resolve()
    raise FileNotFoundError('Cannot locate scripts/AfferentBalance2026')


_live_project = _live_project_root()
sys.path.insert(0, str(_live_project.parents[1]))
from scripts.AfferentBalance2026.experiments.notebook_runtime import (
    refresh_model_modules, resolve_genome_path,
)
# A running Jupyter kernel may still hold the pre-symmetry 54/56-coordinate codecs.
# Refresh dependent loaders together before importing any classes below.
refresh_model_modules()
from scripts.AfferentBalance2026.experiments.D_trajectory.controller import AngleController, load_solution
from scripts.AfferentBalance2026.experiments.simulation import MODEL
if MODEL != "SpikingNN.Afferents/v3-tensile":
    raise RuntimeError("Restart the Jupyter kernel: tensile-torque model v3 is required.")
from scripts.AfferentBalance2026.experiments.D_trajectory.trajectory import DT_S, step_count, stream

# Register locally installed fonts without changing the backend of earlier plots.
for _font_dir in (Path.home() / '.local/share/fonts', Path.home() / '.wine/drive_c/windows/Fonts'):
    for _font_path in _font_dir.rglob('times*.ttf'):
        font_manager.fontManager.addfont(str(_font_path))
_live_font = FONT_FAMILY if any(f.name == FONT_FAMILY for f in font_manager.fontManager.ttflist) else 'DejaVu Serif'
_live_genome = resolve_genome_path(
    _live_project / 'experiments/D_trajectory', GENOME_PATH, variant=GENOME_VARIANT,
)
print("Loaded genome:", _live_genome)
_live_snapshots = (Path(SNAPSHOT_DIR).expanduser().resolve() if SNAPSHOT_DIR is not None
                   else _live_project / 'experiments/D_trajectory/results/live_activity_snapshots')


class RecordingAngleController(AngleController):
    """Record each physical step using the same compiled integrator as AngleController.

    Only recording stride changes. Currents, integration order and state updates
    remain identical. ``last_trace`` includes the block's initial observation.
    """

    def reset(self, initial_angle_deg=0., initial_velocity_rad_s=0., target_angle_deg=0.):
        """Reset model state and discard the previous block's recording."""
        snapshot = super().reset(initial_angle_deg, initial_velocity_rad_s, target_angle_deg)
        self.last_trace = np.empty((0, 26))
        return snapshot

    def advance(self, target_angle_deg=None, duration_s=.01):
        """Advance one held command, recording all 0.1 ms samples and spike events."""
        steps = step_count(duration_s)
        if self._failure:
            raise RuntimeError('Simulation stopped; call reset before advancing again')
        if target_angle_deg is not None:
            self.target_angle_deg = target_angle_deg
        reference = np.zeros((steps+1, 2))
        reference[:, 0] = np.deg2rad(self._target)
        # Reuse the controller's complete hidden state, rather than replaying episodes.
        result = stream(self._physical, self._coefficients, self._mechanics,
                        self._state, self._current, reference, self._tau, 0, 1, self._steps)
        self.last_trace = result[1]
        self._steps += result[4]
        self._failure = result[5]
        return self.snapshot()


class LivePendulumActivitySimulation:
    """Display a continuously evolving afferent controller with an online angle input.

    The asyncio task yields between frames, allowing Jupyter widget callbacks
    to run. Stop/resume preserves every neural, muscle and mechanical state.
    Muscle arrows show tensile forces along tendons toward the fixed anchors.
    Their generalized moments follow M_i = -F_i*dL_i/dq in the corrected model.
    """

    def __init__(self, genome_path, snapshot_dir):
        """Load the genome, warm up Numba, build controls and draw the reset state."""
        ticks = COMMAND_INTERVAL_S / .0001
        if (not np.isfinite([COMMAND_INTERVAL_S, WINDOW_S, FPS, TIME_SCALE,
                             FORCE_SCALE_M_PER_N, MAX_ARROW_M, GRAVITY_MIN_ARROW_M]).all()
                or COMMAND_INTERVAL_S < .0001 or not np.isclose(ticks, round(ticks), rtol=0, atol=1e-7)
                or min(WINDOW_S, FPS, TIME_SCALE, FORCE_SCALE_M_PER_N, MAX_ARROW_M, GRAVITY_MIN_ARROW_M) <= 0
                or GRAVITY_MIN_ARROW_M > MAX_ARROW_M
                or not isinstance(MAX_BLOCKS_PER_FRAME, int) or MAX_BLOCKS_PER_FRAME < 1):
            raise ValueError('Invalid timing or arrow settings; commands must use the 0.1 ms grid')
        if NEURON_VIEW not in ('membrane', 'spikes') or not isinstance(MAX_DISPLAY_POINTS, int) or MAX_DISPLAY_POINTS < 4:
            raise ValueError('Invalid neural view or display point budget')
        self.genome_path = Path(genome_path)
        self.snapshot_dir = Path(snapshot_dir)
        cfg, physical, _, _, _ = load_solution(self.genome_path)
        self.mass, self.length, _, self.gravity = cfg.mechanics
        self.a1, self.a2 = physical[46:48]
        # Both muscles may receive each neuron; color denotes its strongest projection.
        dominant_muscles = np.argmax(physical[48:52].reshape(2, 2), axis=0)
        self.neuron_colors = tuple((FLEXOR_COLOR, EXTENSOR_COLOR)[m] for m in dominant_muscles)
        self.neuron_labels = tuple(f'N{i} ({("Flexor", "Extensor")[m]})'
                                   for i, m in enumerate(dominant_muscles))
        self.controller = RecordingAngleController(self.genome_path, initial_angle_deg=INITIAL_ANGLE_DEG,
                                           initial_velocity_rad_s=INITIAL_SPEED_RAD_S,
                                           target_angle_deg=INITIAL_TARGET_DEG)
        self.controller.advance(INITIAL_TARGET_DEG, COMMAND_INTERVAL_S)  # Compile before timing starts.
        self.controller.reset(INITIAL_ANGLE_DEG, INITIAL_SPEED_RAD_S, INITIAL_TARGET_DEG)
        self.history = deque(maxlen=math.ceil(WINDOW_S / COMMAND_INTERVAL_S) + 3)
        self.activity_history = deque()  # Chunks: time, V0, V1, afferent sums F/E, forces F/E (N).
        self.spike_times = [deque(), deque()]  # Exact event times for the two model neurons.
        self._record_activity()
        self._task, self.running, self.closed = None, False, False
        self.last_error, self.last_snapshot_path = None, None
        self._record()
        self.target_slider = widgets.FloatSlider(
            value=INITIAL_TARGET_DEG, min=-45., max=45., step=TARGET_STEP_DEG,
            description='Цель, °:', continuous_update=True, readout_format='.1f',
            style={'description_width': 'initial'}, layout=widgets.Layout(width='470px'))
        self.start_stop = widgets.Button(description='Старт', icon='play', button_style='success')
        self.reset_button = widgets.Button(description='Сброс', icon='undo')
        self.snapshot_button = widgets.Button(description='Снимок PNG', icon='camera')
        self.neuron_view = widgets.ToggleButtons(
            options=[('Потенциалы, мВ', 'membrane'), ('Растр спайков', 'spikes')],
            value=NEURON_VIEW, description='Нейроны:', style={'description_width': 'initial'})
        self.readout = widgets.HTML()
        self.message = widgets.HTML()
        self.image = widgets.Image(format='png', layout=widgets.Layout(width='100%', max_width='1450px'))
        controls = widgets.HBox([self.start_stop, self.reset_button, self.snapshot_button, self.target_slider],
                                 layout=widgets.Layout(flex_flow='row wrap', align_items='center'))
        self.ui = widgets.VBox([controls, self.neuron_view, self.readout, self.image, self.message])
        self.neuron_view.observe(self._neuron_view_changed, names='value')
        self.target_slider.observe(self._target_changed, names='value')
        self.start_stop.on_click(self._toggle)
        self.reset_button.on_click(self._reset_clicked)
        self.snapshot_button.on_click(self._snapshot_clicked)
        self._make_figure()
        self._draw()
        self._notice('Мышечные времена ≥50 мс: загружен переобученный геном. Метрики проверки приведены в README.')

    def _make_figure(self):
        """Create persistent artists once; later frames only update their data."""
        style = {'font.family': _live_font, 'font.size': FONT_SIZE, 'axes.titlesize': TITLE_SIZE,
                 'axes.labelsize': LABEL_SIZE, 'xtick.labelsize': TICK_SIZE,
                 'ytick.labelsize': TICK_SIZE, 'legend.fontsize': LEGEND_SIZE, 'mathtext.fontset': 'stix'}
        with mpl.rc_context(style):
            self.figure = Figure(figsize=FIGSIZE, dpi=FRAME_DPI, layout='constrained')
            self.canvas = FigureCanvasAgg(self.figure)
            # Fit all four time plots within the existing figure height.
            grid = self.figure.add_gridspec(4, 2, width_ratios=[1., 1.6])
            self.scene = self.figure.add_subplot(grid[:, 0])
            self.chart = self.figure.add_subplot(grid[0, 1])
            self.neuron_chart = self.figure.add_subplot(grid[1, 1], sharex=self.chart)
            self.afferent_chart = self.figure.add_subplot(grid[2, 1], sharex=self.chart)
            self.muscle_chart = self.figure.add_subplot(grid[3, 1], sharex=self.chart)
            ax = self.scene
            extent = max(self.length, self.a1 + .06, self.a2 + MAX_ARROW_M) * 1.12
            ax.set(xlim=(-extent, extent), ylim=(-self.length*.65, self.length*1.28),
                   aspect='equal', title='')
            ax.axis('off')
            ax.plot([-extent*.9, extent*.9], [0, 0], color='#a7a7a7', lw=1, zorder=0)
            ax.plot([0, 0], [0, self.length*1.06], ':', color='#b6b6b6', lw=1)
            self.target_line, = ax.plot([], [], '-', color=TARGET_COLOR, lw=2, zorder=6, label='Target')
            self.flexor_line, = ax.plot([], [], color=FLEXOR_COLOR, lw=2.5, label='Flexor')
            self.extensor_line, = ax.plot([], [], color=EXTENSOR_COLOR, lw=2.5, label='Extensor')
            self.limb_line, = ax.plot([], [], '-o', color=LIMB_COLOR, lw=4, markersize=5, label='Limb')
            ax.plot([0], [0], 'o', color='black', ms=8, zorder=10)
            ax.plot([self.a1], [0], 's', color=FLEXOR_COLOR, ms=5)
            ax.plot([-self.a1], [0], 's', color=EXTENSOR_COLOR, ms=5)
            self.force_arrows = []
            for color in (FLEXOR_COLOR, EXTENSOR_COLOR, '#9b6818'):
                arrow = FancyArrowPatch((0, 0), (.01, 0), arrowstyle='-|>',
                                         mutation_scale=15, lw=2, color=color, zorder=8)
                ax.add_patch(arrow)
                self.force_arrows.append(arrow)
            self.gravity_label = ax.text(0, 0, 'mg', color='#9b6818', fontsize=ANNOTATION_SIZE)
            self.force_text = ax.text(.01, .17, '', transform=ax.transAxes,
                                      fontsize=ANNOTATION_SIZE, va='bottom')
            self.angle_text = ax.text(.02, .96, '', transform=ax.transAxes,
                                      fontsize=ANNOTATION_SIZE, va='top')
            ax.legend(loc='lower left', bbox_to_anchor=(.0, .0), ncol=2, framealpha=.85)
            self.target_trace, = self.chart.plot([], [], '--', color=TARGET_COLOR, lw=2,
                                                 drawstyle='steps-post', label='Target angle')
            self.angle_trace, = self.chart.plot([], [], color=LIMB_COLOR, lw=1.5, label='Limb`s angle')
            self.chart.set(title='', xlabel='',
                            ylabel='Angle, °', xlim=(0, WINDOW_S), ylim=(-55, 55))
            self.voltage_traces, self.spike_traces, self.afferent_traces = [], [], []
            self.muscle_traces = []
            for i, color in enumerate(self.neuron_colors):
                line, = self.neuron_chart.plot([], [], color=color, lw=.8, label=self.neuron_labels[i])
                raster, = self.neuron_chart.plot([], [], '|', color=color, ms=10, ls='none')
                self.voltage_traces.append(line)
                self.spike_traces.append(raster)
            for label, color in (('Flexor', FLEXOR_COLOR), ('Extensor', EXTENSOR_COLOR)):
                line, = self.afferent_chart.plot([], [], color=color, lw=1.2, label=label)
                self.afferent_traces.append(line)
                force, = self.muscle_chart.plot([], [], color=color, lw=1.2, label=label)
                self.muscle_traces.append(force)
            self.neuron_chart.legend(handles=self.voltage_traces, loc='upper right', ncol=2)
            self.afferent_chart.set(ylabel='Ia + II + Ib\n(model units)', ylim=(0, 1))
            self.afferent_chart.legend(loc='upper right', ncol=2)
            self.muscle_chart.set(xlabel='t, s', ylabel='Force, N', ylim=(0, 1))
            self.muscle_chart.legend(loc='upper right', ncol=2)
            self.chart.tick_params(labelbottom=False)
            self.neuron_chart.tick_params(labelbottom=False)
            self.afferent_chart.tick_params(labelbottom=False)
            self.neuron_chart.grid(alpha=.25)
            self.afferent_chart.grid(alpha=.25)
            self.muscle_chart.grid(alpha=.25)
            self._configure_neuron_axis()
            # Bold panel labels stay attached to their respective axes.
            for panel, label in ((self.scene, 'A'), (self.chart, 'B'),
                                 (self.neuron_chart, 'C'), (self.afferent_chart, 'D'),
                                 (self.muscle_chart, 'E')):
                panel.set_title(label, loc='left', fontweight='bold', fontsize=TITLE_SIZE)
            self.chart.grid(alpha=.25)
            self.chart.legend(loc='upper right')
            self.figure.supxlabel('Muscle arrows: tendon tension toward anchors. '
                                   'mg: gravity (enlarged for visibility). Model: v3-tensile.',
                                   fontsize=max(8, ANNOTATION_SIZE-1), fontfamily=_live_font)

    def _configure_neuron_axis(self):
        """Switch between membrane potentials and exact spike events without resetting state."""
        membrane = self.neuron_view.value == 'membrane'
        for voltage, raster in zip(self.voltage_traces, self.spike_traces):
            voltage.set_visible(membrane)
            raster.set_visible(not membrane)
        if membrane:
            self.neuron_chart.set_ylabel('V, mV')
            self.neuron_chart.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=5))
        else:
            self.neuron_chart.set(ylabel='Neuron', ylim=(-.6, 1.6))
            self.neuron_chart.set_yticks([0, 1], labels=['N0', 'N1'])

    def _neuron_view_changed(self, _change):
        """Apply the display mode immediately; keep the same physical trajectory."""
        self._configure_neuron_axis()
        if not self.running:
            self._draw()

    def _record_activity(self, trace=None):
        """Retain voltages, afferent sums, forces and spikes for the configured window."""
        if trace is None:
            state = self.controller.snapshot()
            sums = state['afferents'].reshape(2, 3).sum(axis=1)
            rows = np.array([[state['time_s'], *state['membrane_mV'], *sums, *state['force_N']]])
        else:
            # Skip the block's initial row to avoid repeating samples and spikes.
            post = trace[1:]
            if not len(post):
                return
            rows = np.column_stack((post[:, 0], post[:, 5:7],
                                    post[:, 11:14].sum(axis=1), post[:, 14:17].sum(axis=1),
                                    post[:, 3:5]))  # Actual flexor/extensor tension in newtons.
            for i in range(2):
                self.spike_times[i].extend(post[post[:, 19+i] > 0, 0])
        self.activity_history.append(rows)
        cutoff = rows[-1, 0]-WINDOW_S
        while len(self.activity_history) > 1 and self.activity_history[0][-1, 0] < cutoff:
            self.activity_history.popleft()
        # Preserve at most one sample preceding the left edge for continuous curves.
        first = self.activity_history[0]
        skip = max(0, int(np.searchsorted(first[:, 0], cutoff))-1)
        if skip:
            self.activity_history[0] = first[skip:].copy()
        for events in self.spike_times:
            while events and events[0] < cutoff:
                events.popleft()

    @staticmethod
    def _display_envelope(times, values):
        """Reduce drawing cost with time-ordered extrema; never decimate the recorded data.

        Each bin contributes its minimum and maximum. Peaks therefore remain
        visible even when they last one integration step. The spike raster uses
        all event times directly and does not use this display reduction.
        """
        if len(values) <= MAX_DISPLAY_POINTS:
            return times, values
        width = math.ceil(len(values) / max(1, (MAX_DISPLAY_POINTS-4)//2))
        groups = len(values)//width
        blocks = values[:groups*width].reshape(groups, width)
        bases = np.arange(groups)*width
        indices = np.column_stack((bases+blocks.argmin(axis=1), bases+blocks.argmax(axis=1))).ravel()
        if groups*width < len(values):
            tail = values[groups*width:]
            indices = np.r_[indices, groups*width+tail.argmin(), groups*width+tail.argmax()]
        indices = np.unique(np.r_[0, indices, len(values)-1])
        return times[indices], values[indices]

    def geometry(self, state=None):
        """Return diagram geometry and force vectors consistent with model moments.

        Positive theta is clockwise from the upward vertical: r=(sin(theta), cos(theta)).
        Tendon lengths match L(q) at q=theta-pi/2. Positive tension pulls the
        attachment toward its anchor; moment is force dotted with dr/dtheta.
        Gravity acts downward at the midpoint of the uniform rod.
        """
        state = self.controller.snapshot() if state is None else state
        theta = np.deg2rad(state['angle_deg'])
        direction = np.array([np.sin(theta), np.cos(theta)])
        tangent = np.array([np.cos(theta), -np.sin(theta)])
        attachment = self.a2*direction
        anchors = np.array([[self.a1, 0.], [-self.a1, 0.]])
        lengths = np.linalg.norm(anchors-attachment, axis=1)
        # A muscle applies tension toward its fixed anchor, never a pushing force.
        muscle_vectors = state['force_N'][:, None]*(anchors-attachment)/lengths[:, None]
        moments = muscle_vectors @ (self.a2*tangent)
        gravity_vector = np.array([0., -self.mass*self.gravity])
        return dict(direction=direction, tangent=tangent, attachment=attachment, anchors=anchors,
                    lengths=lengths, moments=moments, muscle_vectors=muscle_vectors,
                    gravity_vector=gravity_vector, center=.5*self.length*direction)

    def _record(self):
        """Retain the rolling window and one preceding sample for a continuous boundary."""
        s = self.controller.snapshot()
        row = (s['time_s'], s['target_angle_deg'], s['angle_deg'])
        if self.history and abs(self.history[-1][0]-row[0]) < 1e-12:
            self.history[-1] = row
        else:
            self.history.append(row)
        while len(self.history) > 2 and self.history[1][0] < row[0]-WINDOW_S:
            self.history.popleft()

    def _advance_block(self):
        """Advance one command interval; no neuron, muscle or limb state is reset."""
        s = self.controller.advance(float(self.target_slider.value), COMMAND_INTERVAL_S)
        if s['failure'] == 'nonfinite':
            raise FloatingPointError('Нечисловое состояние; требуется сброс.')
        self._record()
        self._record_activity(self.controller.last_trace)
        return s

    def _draw_arrow(self, artist, origin, force, min_length=0.):
        """Apply visual length limits while preserving force direction and physical values."""
        delta = FORCE_SCALE_M_PER_N*np.asarray(force)
        length = np.linalg.norm(delta)
        artist.set_visible(length > 1e-10)
        if length > 1e-10:
            delta *= np.clip(length, min_length, MAX_ARROW_M)/length
        artist.set_positions(origin, origin+delta)
        return delta

    def _draw(self):
        """Render the real current state and rolling traces into the notebook image widget."""
        s = self.controller.snapshot()
        if not np.isfinite([s['angle_deg'], s['velocity_rad_s'], *s['force_N']]).all():
            return  # Keep the last valid frame if the numerical model fails.
        g = self.geometry(s)
        target = np.deg2rad(self.target_slider.value)
        # Extend the target slightly so it stays visible when the limb reaches it.
        target_tip = 1.12*self.length*np.array([np.sin(target), np.cos(target)])
        limb_tip = self.length*g['direction']
        self.target_line.set_data([0, target_tip[0]], [0, target_tip[1]])
        self.limb_line.set_data([0, limb_tip[0]], [0, limb_tip[1]])
        for line, anchor in zip((self.flexor_line, self.extensor_line), g['anchors']):
            line.set_data([anchor[0], g['attachment'][0]], [anchor[1], g['attachment'][1]])
        for arrow, force in zip(self.force_arrows[:2], g['muscle_vectors']):
            self._draw_arrow(arrow, g['attachment'], force)
        gravity_delta = self._draw_arrow(
            self.force_arrows[2], g['center'], g['gravity_vector'], min_length=GRAVITY_MIN_ARROW_M)
        self.gravity_label.set_position(g['center']+.5*gravity_delta+np.array([.012, 0.]))
        ff, fe = s['force_N']
        mf, me = g['moments']
        self.force_text.set_text(f'Силы мышц: Fсг = {ff:.1f} Н; Fразг = {fe:.1f} Н\n'
                                 f'Моменты: Mсг = {mf:+.3f} Н·м; Mразг = {me:+.3f} Н·м')
        self.angle_text.set_text(f't = {s["time_s"]:.2f} с\nθ* = {self.target_slider.value:+.1f}°\n'
                                 f'θ = {s["angle_deg"]:+.1f}°')
        data = np.asarray(self.history)
        self.target_trace.set_data(data[:, 0], data[:, 1])
        self.angle_trace.set_data(data[:, 0], data[:, 2])
        right = max(WINDOW_S, s['time_s'])
        self.chart.set_xlim(right-WINDOW_S, right)
        limit = max(55., float(np.max(abs(data[:, 1:]))+5.))
        self.chart.set_ylim(-limit, limit)
        activity = np.concatenate(tuple(self.activity_history))
        for i, (voltage, raster, afferent, muscle) in enumerate(zip(
                self.voltage_traces, self.spike_traces, self.afferent_traces, self.muscle_traces)):
            voltage.set_data(*self._display_envelope(activity[:, 0], activity[:, 1+i]))
            events = np.asarray(self.spike_times[i])
            raster.set_data(events, np.full(len(events), i))
            afferent.set_data(*self._display_envelope(activity[:, 0], activity[:, 3+i]))
            muscle.set_data(*self._display_envelope(activity[:, 0], activity[:, 5+i]))
        if self.neuron_view.value == 'membrane':
            self.neuron_chart.set_ylim(min(-90., float(activity[:, 1:3].min())-5.),
                                       max(35., float(activity[:, 1:3].max())+5.))
        self.afferent_chart.set_ylim(0., max(1., 1.1*float(activity[:, 3:5].max())))
        self.muscle_chart.set_ylim(0., max(1., 1.1*float(activity[:, 5:7].max())))
        state_label = 'Работает' if self.running else 'Пауза'
        self.readout.value = (f'<div style="font-family:{escape(_live_font)};font-size:{WIDGET_FONT_SIZE}px">'
                              f'<b>{state_label}</b> · t = {s["time_s"]:.2f} с · '
                              f'цель = {self.target_slider.value:+.1f}° · угол = {s["angle_deg"]:+.1f}° · '
                              f'скорость = {s["velocity_rad_s"]:+.2f} рад/с</div>')
        # A PNG widget avoids backend switching and works in JupyterLab and Notebook.
        buffer = BytesIO()
        self.canvas.print_png(buffer)
        self.image.value = buffer.getvalue()

    def _notice(self, text, error=False):
        """Show a compact status or saved-image path, with HTML escaping."""
        color = '#a32020' if error else '#555555'
        self.message.value = (f'<span style="color:{color};font-size:{WIDGET_FONT_SIZE}px">'
                               f'{escape(str(text))}</span>')

    def _target_changed(self, change):
        """Update the command immediately while preserving the actual limb state."""
        self.controller.target_angle_deg = float(change['new'])
        self._record()
        if not self.running:
            self._draw()

    def _toggle(self, _button):
        """Handle the combined start/stop button."""
        self.stop() if self.running else self.start()

    def start(self):
        """Start or resume without resetting state or counting time spent paused."""
        if self.closed or self.running:
            return
        if self.controller.snapshot()['failure'] != 'none':
            self._notice('Расчёт остановлен после падения. Нажмите «Сброс».', error=True)
            return
        self.running = True
        self._wall_origin = time.perf_counter()
        self._sim_origin = self.controller.snapshot()['time_s']
        self.start_stop.description, self.start_stop.icon, self.start_stop.button_style = 'Стоп', 'pause', 'warning'
        self._notice('Меняйте целевой угол ползунком. Снимок сохраняет все пять панелей и ставит расчёт на паузу.')
        self._task = asyncio.get_running_loop().create_task(self._run())

    async def _run(self):
        """Track wall time with bounded catch-up and yield regularly to widget callbacks."""
        task = asyncio.current_task()
        try:
            while self.running and self._task is task:
                started = time.perf_counter()
                desired_elapsed = (started-self._wall_origin)*TIME_SCALE
                actual_elapsed = self.controller.snapshot()['time_s']-self._sim_origin
                blocks = min(MAX_BLOCKS_PER_FRAME,
                             max(0, int((desired_elapsed-actual_elapsed+1e-10)/COMMAND_INTERVAL_S)))
                for _ in range(blocks):
                    s = self._advance_block()
                    if s['failure'] != 'none':
                        self.running = False
                        self._notice('Конечность вышла за ±90°. Расчёт остановлен; нажмите «Сброс».', error=True)
                        break
                self._draw()
                # Every 0.1 ms physical step is retained. Slow rendering can cause lag,
                # but elapsed simulation time is never fabricated or skipped forward.
                await asyncio.sleep(max(.001, 1/FPS-(time.perf_counter()-started)))
        except asyncio.CancelledError:
            pass
        except Exception as error:
            self.last_error = error
            self._notice(f'Расчёт остановлен: {error}', error=True)
        finally:
            if self._task is task:
                self._task, self.running = None, False
                self.start_stop.description, self.start_stop.icon, self.start_stop.button_style = 'Старт', 'play', 'success'
                self._draw()

    def stop(self):
        """Freeze the model and chart; the next start resumes from this exact state."""
        self.running = False
        task, self._task = self._task, None
        if task is not None and not task.done():
            task.cancel()
        self.start_stop.description, self.start_stop.icon, self.start_stop.button_style = 'Старт', 'play', 'success'
        if not self.closed:
            self._draw()

    def _reset_clicked(self, _button):
        """Reset the physical state and history explicitly, retaining the current slider target."""
        self.stop()
        self.controller.reset(INITIAL_ANGLE_DEG, INITIAL_SPEED_RAD_S, float(self.target_slider.value))
        self.history.clear()
        self.activity_history.clear()
        for events in self.spike_times:
            events.clear()
        self._record_activity()
        self.last_error = None
        self._record()
        self._draw()
        self._notice('Начальное состояние восстановлено. Нажмите «Старт».')

    def save_snapshot(self):
        """Pause and save a high-resolution PNG of the schematic and all four rolling plots."""
        self.stop()
        self.snapshot_dir.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
        path = self.snapshot_dir / f'pendulum_activity_{stamp}_t{self.controller.snapshot()["time_s"]:.2f}s.png'
        self.figure.savefig(path, dpi=SNAPSHOT_DPI, bbox_inches='tight')
        self.last_snapshot_path = path
        self._notice(f'Пауза. Снимок сохранён: {path}')
        return path

    def _snapshot_clicked(self, _button):
        """Report snapshot errors in the widget instead of losing the button callback."""
        try:
            self.save_snapshot()
        except Exception as error:
            self._notice(f'Не удалось сохранить снимок: {error}', error=True)

    def close(self):
        """Cancel the background task and release this cell's old widget instance."""
        if self.closed:
            return
        self.stop()
        self.closed = True
        self.neuron_view.unobserve(self._neuron_view_changed, names='value')
        self.target_slider.unobserve(self._target_changed, names='value')
        for widget in (self.target_slider, self.start_stop, self.reset_button, self.snapshot_button,
                       self.neuron_view, self.readout, self.message, self.image, self.ui):
            widget.close()
        self.figure.clear()


# Rerunning the cell replaces its previous animation instead of starting another task.
_previous_live_pendulum_activity = globals().get('live_pendulum_activity')
if _previous_live_pendulum_activity is not None:
    _previous_live_pendulum_activity.close()
live_pendulum_activity = LivePendulumActivitySimulation(_live_genome, _live_snapshots)
display(live_pendulum_activity.ui)
if AUTO_START:
    live_pendulum_activity.start()
# Optional programmatic controls: live_pendulum_activity.stop(), .start(), .save_snapshot().


Loaded genome: /home/nickolai/neuromorph/CPG/equipment/SpikingNN/scripts/AfferentBalance2026/experiments/D_trajectory/best_genome_slow_muscle_mirrow.json


Очень странные колебания, которые возникат изза пачечного режима...
